Code BERT SLM Training

In [ ]:
!pip install -q pytorch-lightning transformers datasets peft torchmetrics scikit-learn

In [ ]:
import os
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)
from peft import get_peft_model, LoraConfig, TaskType
import torchmetrics

# --- CONFIGURATION ---
MODEL_CHECKPOINT = "microsoft/codebert-base"
BATCH_SIZE = 32
MAX_LENGTH = 256
LR = 2e-4
EPOCHS = 3

# --- 1. LIGHTNING DATA MODULE ---
class SuricataDataModule(pl.LightningDataModule):
    def __init__(self, train_file, val_file, tokenizer_name, batch_size=32, max_length=256):
        super().__init__()
        self.train_file = train_file
        self.val_file = val_file
        self.tokenizer_name = tokenizer_name
        self.batch_size = batch_size
        self.max_length = max_length

    def setup(self, stage=None):
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_name)
        
        # Load Datasets
        dataset = load_dataset("json", data_files={"train": self.train_file, "validation": self.val_file})
        
        def preprocess_function(examples):
            return self.tokenizer(
                examples["text"], 
                truncation=True, 
                padding="max_length", 
                max_length=self.max_length
            )
        
        tokenized = dataset.map(preprocess_function, batched=True)

        # FIX 1: Ensure column is named 'labels' (Transformers expectation)
        if "label" in tokenized["train"].column_names:
            tokenized = tokenized.rename_column("label", "labels")
        
        # Set format for PyTorch
        tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
        
        self.tokenized_datasets = tokenized
        self.data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

    def train_dataloader(self):
        return DataLoader(
            self.tokenized_datasets["train"],
            batch_size=self.batch_size,
            collate_fn=self.data_collator,
            shuffle=True,
            num_workers=os.cpu_count() or 2
        )

    def val_dataloader(self):
        return DataLoader(
            self.tokenized_datasets["validation"],
            batch_size=self.batch_size,
            collate_fn=self.data_collator,
            num_workers=os.cpu_count() or 2
        )

# --- 2. LIGHTNING MODULE (MODEL) ---
class SuricataClassifier(pl.LightningModule):
    def __init__(self, model_name, lr=2e-4, num_labels=2):
        super().__init__()
        self.save_hyperparameters()
        
        self.base_model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label={0: "INVALID", 1: "VALID"},
            label2id={"INVALID": 0, "VALID": 1}
        )
        
        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=16,
            lora_alpha=32,
            lora_dropout=0.1
        )
        self.model = get_peft_model(self.base_model, peft_config)
        
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=2)
        self.f1 = torchmetrics.F1Score(task="multiclass", num_classes=2, average='macro')

    def forward(self, **inputs):
        return self.model(**inputs)

    def training_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        preds = torch.argmax(outputs.logits, dim=1)
        
        self.accuracy(preds, batch["labels"])
        self.f1(preds, batch["labels"])
        
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", self.accuracy, prog_bar=True)
        self.log("val_f1", self.f1, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        
        # FIX 2: Use estimated_stepping_batches to avoid circular dataloader calls
        total_steps = self.trainer.estimated_stepping_batches
        warmup_steps = int(0.1 * total_steps)
        
        scheduler = get_linear_schedule_with_warmup(
            optimizer, 
            num_warmup_steps=warmup_steps, 
            num_training_steps=total_steps
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
            }
        }

# --- 3. EXECUTION ---
if __name__ == "__main__":
    if not os.path.exists("train.jsonl") or not os.path.exists("validation.jsonl"):
        print("⚠️ Error: JSONL files not found.")
    else:
        data_module = SuricataDataModule(
            train_file="train.jsonl",
            val_file="validation.jsonl",
            tokenizer_name=MODEL_CHECKPOINT,
            batch_size=BATCH_SIZE
        )

        model = SuricataClassifier(model_name=MODEL_CHECKPOINT, lr=LR)

        trainer = pl.Trainer(
            max_epochs=EPOCHS,
            accelerator="auto",
            devices=1,
            precision="16-mixed",
            log_every_n_steps=10
        )

        print("Starting Lightning Training...")
        trainer.fit(model, datamodule=data_module)

        print("Saving model...")
        model.model.save_pretrained("./bert_suricata_model_lightning")
        data_module.tokenizer.save_pretrained("./bert_suricata_model_lightning")
        print("Done!")

Validate Suricata Rules using SLM Model

In [ ]:
# --- 5. INFERENCE / TESTING ---
from peft import PeftModel, PeftConfig
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Configuration
WRITE_OUTPUT = False  # Set to True to write valid.rules and invalid.rules files
INPUT_FILE = "input.rules"

# Load Base Model & Tokenizer (MUST MATCH TRAINING MODEL)
# Note: Changed from 'distilroberta' to 'microsoft/codebert-base' to match training
base_model_name = "microsoft/codebert-base" 
tokenizer = AutoTokenizer.from_pretrained("./bert_suricata_model_lightning") # Load saved tokenizer
base_model = AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2)

# Load Trained Adapters
inference_model = PeftModel.from_pretrained(base_model, "./bert_suricata_model_lightning")
inference_model.to("cuda" if torch.cuda.is_available() else "cpu")
inference_model.eval()

def check_rule(rule_text):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(rule_text, return_tensors="pt", truncation=True, max_length=256).to(device)
    
    with torch.no_grad():
        logits = inference_model(**inputs).logits
    
    prediction = torch.argmax(logits, dim=1).item()
    return "VALID" if prediction == 1 else "INVALID"

def process_rules_file(input_file):
    """Read rules from input file and classify them as valid or invalid."""
    valid_rules = []
    invalid_rules = []
    
    try:
        with open(input_file, 'r') as f:
            for line in f:
                rule = line.strip()
                # Skip empty lines and comments
                if not rule or rule.startswith('#'):
                    continue
                
                status = check_rule(rule)
                if status == "VALID":
                    valid_rules.append(rule)
                else:
                    invalid_rules.append(rule)
    except FileNotFoundError:
        print(f"Error: Input file '{input_file}' not found.")
        return [], []
    
    return valid_rules, invalid_rules

def main():
    print(f"Processing rules from '{INPUT_FILE}'...")
    valid_rules, invalid_rules = process_rules_file(INPUT_FILE)
    
    # Output valid rules section
    print("\n" + "=" * 60)
    print("VALID RULES")
    print("=" * 60)
    if valid_rules:
        for rule in valid_rules:
            print(rule)
    else:
        print("No valid rules found.")
    
    # Output invalid rules section
    print("\n" + "=" * 60)
    print("INVALID RULES")
    print("=" * 60)
    if invalid_rules:
        for rule in invalid_rules:
            print(rule)
    else:
        print("No invalid rules found.")
    
    # Summary
    print("\n" + "=" * 60)
    print(f"Summary: {len(valid_rules)} valid, {len(invalid_rules)} invalid")
    print("=" * 60)
    
    # Write to files if WRITE_OUTPUT is enabled
    if WRITE_OUTPUT:
        with open("valid.rules", 'w') as f:
            for rule in valid_rules:
                f.write(rule + '\n')
        print(f"\nValid rules written to 'valid.rules'")
        
        with open("invalid.rules", 'w') as f:
            for rule in invalid_rules:
                f.write(rule + '\n')
        print(f"Invalid rules written to 'invalid.rules'")

if __name__ == "__main__":
    main()